# Assignment 2

 **This was supoosed to be 2 assignment but is a big assignment So go slow and learn what you are doing have fun**


# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we Doing only for pdf but in final project we will do for pdf,csv,xlsx,docx,txt.
if you want you can practise to extract data from one more file format i would love to see you do this.

In [1]:

import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5896\253017937.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\Users\Lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = [] 
    pdf_paths = list(Path(pdf_directory).glob('**/*.pdf'))
    
    for pdf_path in pdf_paths:
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()
         
        for doc in docs:
            doc.metadata['source_file'] = pdf_path.name
            doc.metadata['file_type'] = 'pdf'
            
        all_documents.extend(docs)
        
    return all_documents

In [ ]:
 
all_pdf_documents = process_all_pdfs("data")
 
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-01T10:57:35-07:00', 'moddate': '2026-06-01T10:57:35-07:00', 'source': "data\\Quant ITP'26.pdf", 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': "Quant ITP'26.pdf", 'file_type': 'pdf'}, page_content='QUANT INTERN \nPREPARATION'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-01T10:57:35-07:00', 'moddate': '2026-06-01T10:57:35-07:00', 'source': "data\\Quant ITP'26.pdf", 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': "Quant ITP'26.pdf", 'file_type': 'pdf'}, page_content='ABOUT THE PANEL \nAnirvan Tyagi (EE) – Quant Research Intern, Trexquant\n9911130702, anirvant23@iitk.ac.in\nSahil Pandey (EE) – QR Intern, Graviton Research Capital LLP\n8302738127, sahilp23@iitk.ac.in\nYatharth Dangi (CSE) – Quant Intern, Quadeye | Ex-Intern, Google\n6377153842, yatharth23@iitk.ac.in\n2'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF'

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since Large Language Models (LLMs) have a limited context window (input size limit) and retrieve information more accurately from smaller context blocks, we must split large documents. In this assignment, you need to use **RecursiveCharacterTextSplitter** to split loaded documents into smaller paragraphs with proper overlap to maintain context between boundary lines.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
    
    return chunks

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split 14 documents into 14 chunks.


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-01T10:57:35-07:00', 'moddate': '2026-06-01T10:57:35-07:00', 'source': "data\\Quant ITP'26.pdf", 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': "Quant ITP'26.pdf", 'file_type': 'pdf'}, page_content='QUANT INTERN \nPREPARATION'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-01T10:57:35-07:00', 'moddate': '2026-06-01T10:57:35-07:00', 'source': "data\\Quant ITP'26.pdf", 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': "Quant ITP'26.pdf", 'file_type': 'pdf'}, page_content='ABOUT THE PANEL \nAnirvan Tyagi (EE) – Quant Research Intern, Trexquant\n9911130702, anirvant23@iitk.ac.in\nSahil Pandey (EE) – QR Intern, Graviton Research Capital LLP\n8302738127, sahilp23@iitk.ac.in\nYatharth Dangi (CSE) – Quant Intern, Quadeye | Ex-Intern, Google\n6377153842, yatharth23@iitk.ac.in\n2'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF'

# Part 3: Embedding

Embedding is the process of converting text blocks into numerical vector representations. These vectors capture the semantic meaning of the text. Sentences that are semantically similar will be closer to each other in the vector space. We use pre-trained sentence transformer models (like 'all-MiniLM-L6-v2') to convert text chunks and queries into embeddings.

---



from sentence_transformers import SentenceTransformer

Imports the embedding model class.

This library:

downloads pretrained transformer models
converts text → embeddings

Built on top of:

HuggingFace transformers
PyTorch

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid #to get each chunk a unique id after embedding
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        print(f"Loading SentenceTransformer model: {self.model_name}...")
        self.model = SentenceTransformer(self.model_name)
        print(f"Successfully loaded! Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if self.model is None:
            raise ValueError("Model is not loaded.")
        
        return self.model.encode(texts, show_progress_bar=True)


# Part 4: Vector DB

Vector Database (Vector DB) is a specialized database designed to store and query high-dimensional vector embeddings efficiently. It allows us to persist our document chunks along with their computed embeddings and perform fast search operations. Here, we use ChromaDB, which runs locally and stores documents persistently in a directory.

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF Document chunks"}
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings.")
            
        ids = []
        metadatas = []
        texts = []
        embeddings_list = embeddings.tolist()
        
        for i, doc in enumerate(documents):
            
            doc_id = uuid.uuid4().hex[:8]
            ids.append(doc_id)
             
            meta = doc.metadata.copy()
            meta['doc_index'] = i
            meta['content_length'] = len(doc.page_content)
            metadatas.append(meta)
             
            texts.append(doc.page_content)
            
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=texts
        )

In [10]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-01T10:57:35-07:00', 'moddate': '2026-06-01T10:57:35-07:00', 'source': "data\\Quant ITP'26.pdf", 'total_pages': 14, 'page': 0, 'page_label': '1', 'source_file': "Quant ITP'26.pdf", 'file_type': 'pdf'}, page_content='QUANT INTERN \nPREPARATION'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-06-01T10:57:35-07:00', 'moddate': '2026-06-01T10:57:35-07:00', 'source': "data\\Quant ITP'26.pdf", 'total_pages': 14, 'page': 1, 'page_label': '2', 'source_file': "Quant ITP'26.pdf", 'file_type': 'pdf'}, page_content='ABOUT THE PANEL \nAnirvan Tyagi (EE) – Quant Research Intern, Trexquant\n9911130702, anirvant23@iitk.ac.in\nSahil Pandey (EE) – QR Intern, Graviton Research Capital LLP\n8302738127, sahilp23@iitk.ac.in\nYatharth Dangi (CSE) – Quant Intern, Quadeye | Ex-Intern, Google\n6377153842, yatharth23@iitk.ac.in\n2'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF'

## convert text to embeddings


In [11]:
texts=[doc.page_content for doc in chunks]
print(f"Number of chunks: {len(chunks)}")
print(f"Number of texts: {len(texts)}")
texts

Number of chunks: 14
Number of texts: 14


['QUANT INTERN \nPREPARATION',
 'ABOUT THE PANEL \nAnirvan Tyagi (EE) – Quant Research Intern, Trexquant\n9911130702, anirvant23@iitk.ac.in\nSahil Pandey (EE) – QR Intern, Graviton Research Capital LLP\n8302738127, sahilp23@iitk.ac.in\nYatharth Dangi (CSE) – Quant Intern, Quadeye | Ex-Intern, Google\n6377153842, yatharth23@iitk.ac.in\n2',
 'WHY QUANT?\nLucrative field.\nQuant has earned a well-deserved reputation for being one of the most \nprofitable fields currently, and rightly so.\nDynamic Industry\nThis is one of the few fields, with something for everybody. There has been a \nboost in the no. of Indian Quant firms as well, which provides a promise of \nfuture opportunities.\nHigh Focus on Innovation\nQuant involves Math, Strategies, Systems, Algorithms. It respects novel ideas, \nand will involve experiences very different from those developed in college.\n3',
 'WHY NOT QUANT?\n• It is pretty competitive\n• The working hours are long\n• College Education hasn’t given adequate int

In [ ]:
 
embedding_manager = EmbeddingManager()
vectorstore = VectorStore(collection_name="my_pdf_data", persist_directory="./vector_db")

Loading SentenceTransformer model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7402.56it/s]


Successfully loaded! Embedding dimension: 384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5896\3796264850.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Successfully loaded! Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [ ]:
 

embeddings=embedding_manager.generate_embeddings(texts)
 
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  6.04it/s]


# Part 5: Query Retrieval

Query Retrieval starts with the user entering a natural language query. We must convert this query into the same embedding space using our embedding manager. This encoded query is then sent to the vector database to retrieve raw results.

---

# Part 6: Similarity Search

Similarity Search is the mathematical calculation (such as Cosine Similarity) used by the vector store to compare the query embedding with document embeddings. It ranks and returns the top_k most similar documents. We can filter results by a minimum similarity score (score_threshold) to keep only relevant context.


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """ 
        query_embedding = self.embedding_manager.generate_embeddings([query])[0].tolist()
         
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        
        retrieved_docs = []
        
        distances = results['distances'][0]
        docs = results['documents'][0]
        metas = results['metadatas'][0]
        doc_ids = results['ids'][0]
        
        for rank, (doc_id, text, meta, dist) in enumerate(zip(doc_ids, docs, metas, distances)):
             
            similarity_score = 1 - dist
             
            if similarity_score >= score_threshold:
                
                retrieved_docs.append({
                    'id': doc_id,
                    'content': text,
                    'metadata': meta,
                    'similarity_score': similarity_score,
                    'distance': dist,
                    'rank': rank + 1
                })
                
        return retrieved_docs

In [ ]:
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)
 
my_query = "What  is birthday paradox?"
 
search_results = rag_retriever.retrieve(my_query, top_k=3)
 
print(f"--- Top matches for your query ---\n")
for i, result in enumerate(search_results):
    print(f"Match {i+1} (Confidence Score: {result['similarity_score']:.2f}):")
    print(f"{result['content'][:300]}...\n")  
    print("-" * 50)

Batches: 100%|██████████| 1/1 [00:00<00:00, 83.35it/s]

--- Top matches for your query ---



# Part 7: Prompting and Calling LLM

Prompting and Calling LLM is the synthesis step of RAG. We take the retrieved contexts, format them into a structured prompt alongside the user's original query, and pass them to the Large Language Model (LLM) to generate a grounded, factually accurate response.


In [19]:
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([res['content'] for res in results])
    
    prompt = f"Use the following context to answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    
    response = llm.invoke(prompt)
    
    return response.content if hasattr(response, 'content') else str(response)

In [17]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv() 

real_llm = ChatGroq(
    model="llama-3.1-8b-instant", 
    temperature=0
)
 
my_query = "What is the interview process for Graviton Research Capital?"
print(f"Asking: {my_query}\n")
print("-" * 50)
 
answer = rag_simple(my_query, rag_retriever, real_llm)

print("Actual AI Answer:")
print(answer)

Asking: What is the interview process for Graviton Research Capital?

--------------------------------------------------


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.35it/s]


Actual AI Answer:
The interview process for Graviton Research Capital consists of the following steps:

1. **Offline, On-paper test**: This is a 2-hour test in the presence of the company's HR, covering subjective questions on general maths, PMI, probability, PIE, etc.

2. **3 Interview Rounds**: Although the details of the rounds are not explicitly mentioned, the context provides information about the first two rounds and the third round.

3. **Round 1 & 2**: These rounds focus on Probability, Puzzles, and Reasoning, lasting around 30-40 minutes each. The interviewers check not just the answer but also the approach taken to solve the problem. They may provide hints if the candidate is not getting to the answer. The questions cover a mix of standard probability questions and medium-hard difficulty puzzles, as well as some questions about the candidate's resume and campus activities.

4. **Round 3**: This round is focused on the candidate's campus experience and lasts around 10-20 minut

# Part 8: Advanced RAG (Optional)

Advanced RAG includes sophisticated pipeline elements such as streaming responses, citation tracking, interaction history (conversational memory), response summarization, and score-based filtering to make the application robust and production-ready.


In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # TODO: Implement advanced RAG retrieval and generation.
    # 1. Retrieve top_k documents using score threshold min_score.
    # 2. Parse sources including source_file, page, similarity_score, and a content preview.
    # 3. Determine confidence (e.g. max similarity score of retrieved docs).
    # 4. Invoke LLM with formatted prompt combining context and query.
    # 5. Return dict containing 'answer', 'sources', 'confidence' (and 'context' if return_context is True).
    pass


In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # TODO: Implement the AdvancedRAGPipeline query logic:
        # 1. Retrieve documents using self.retriever.
        # 2. Parse sources/metadata and construct the context.
        # 3. If stream=True, simulate streaming by printing the prompt in chunks.
        # 4. Generate the answer using self.llm.
        # 5. Construct citations list and append it to the answer.
        # 6. If summarize=True, call the LLM to get a 2-sentence summary of the answer.
        # 7. Append the question, answer, sources, and summary to self.history.
        # 8. Return dictionary containing question, answer (with citations), sources, summary, and history.
        pass
